# DPO training for the Qwen2.5-3B summarization adapter (Colab)

1. **Runtime → Change runtime type → GPU** (T4 works; L4/A100 is much faster).
2. Add your Hugging Face token under **🔑 Secrets** (left sidebar) as `HF_TOKEN` and enable notebook access. *(Optional — Qwen2.5-3B and the dataset are public, but it avoids rate limits.)*
3. Run the cells top to bottom. The DPO adapter is copied to Google Drive at the end so it survives the session.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO = "https://github.com/thitrantn3/qwen-summarization-dpo.git"
BRANCH = "dpo-training"  # change to "main" after merging
%cd /content
!rm -rf qwen-summarization-dpo && git clone -q -b $BRANCH $REPO
%cd /content/qwen-summarization-dpo
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -U "trl>=0.19" peft accelerate datasets rouge-score evaluate
# Colab ships torchao 0.10, which recent peft/transformers reject on import; we don't use it
!pip uninstall -q -y torchao

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secrets")
except Exception:
    print("No HF_TOKEN secret — continuing anonymously")

## 3. Download the DPO dataset
`--num` controls how many pairs to use (the full set is 100k — start smaller).

In [ ]:
NUM = 5000
!python load_hf_dpo_data.py --num {NUM} --out_dir dpo_data/doc_sum
!wc -l dpo_data/doc_sum/*.jsonl

## 4. Train DPO
On a T4, ~5k pairs for 1 epoch takes a few hours. Lower `NUM` for a quick test run.

In [ ]:
!python train_dpo.py \
    --train_file dpo_data/doc_sum/train.jsonl \
    --val_file   dpo_data/doc_sum/val.jsonl \
    --epochs 1 --lr 5e-6 --beta 0.1

## 5. Save the adapter to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/qwen-summarization-dpo
!cp -r dpo-text-summary /content/drive/MyDrive/qwen-summarization-dpo/
!ls /content/drive/MyDrive/qwen-summarization-dpo/dpo-text-summary

## 6. Quick test: SFT vs DPO

In [ ]:
import torch, json
from model import load_dpo_model

model, tokenizer = load_dpo_model()
row = json.loads(open("dpo_data/doc_sum/val.jsonl").readline())
enc = tokenizer(row["prompt"], return_tensors="pt").to(model.device)

def gen():
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=400, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== DPO ===\n" + gen())
with model.disable_adapter():
    print("\n=== SFT (adapter off) ===\n" + gen())
print("\n=== Dataset 'chosen' ===\n" + row["chosen"])

## 7. Evaluate base vs SFT vs DPO
Scores 200 held-out prompts per model (ROUGE vs the preferred summary, how many source numbers are kept, length/formatting). Lower `--num` for a faster run.

In [ ]:
for m in ["base", "sft", "dpo"]:
    !python evaluate_dpo.py --model {m} --num 200
!python evaluate_dpo.py --compare

In [ ]:
!cp -r results /content/drive/MyDrive/qwen-summarization-dpo/